# Apache Spark RDD Introduction

This notebook uses Apache Spark 3.5.9 in standalone mode. It does not use Hadoop.

You will learn how to:

- create an RDD from hardcoded Python values
- use `map` and `filter` transformations
- run actions such as `collect`, `count`, `take`, and `reduce`
- inspect partitions with `glom`
- connect partitions, tasks, stages, and jobs
- inspect RDD lineage

## 1. Start Spark before running the notebook

In a WSL terminal, start the standalone master and one worker:

```bash
/opt/spark/sbin/start-master.sh
/opt/spark/sbin/start-worker.sh "spark://$(hostname):7077"
jps
```

Open `http://localhost:8080` and confirm that one worker is listed. Keep that page open while running the action cells.

The notebook reads `SPARK_MASTER` when it is set. Otherwise, it connects to `spark://<current-hostname>:7077`.

In [ ]:
import os
import socket

from pyspark.sql import SparkSession

master_url = os.environ.get(
    "SPARK_MASTER",
    f"spark://{socket.gethostname()}:7077",
)

spark = (
    SparkSession.builder
    .appName("D280-Spark-RDD-Introduction")
    .master(master_url)
    .config("spark.default.parallelism", "4")
    .config("spark.ui.showConsoleProgress", "true")
    .getOrCreate()
)

sc = spark.sparkContext
sc.setLogLevel("WARN")

print("Spark version :", spark.version)
print("Spark master  :", sc.master)
print("Application ID:", sc.applicationId)
print("Spark UI      :", sc.uiWebUrl)

The application shown at `http://localhost:8080` is the driver created by this notebook. The application UI printed above is normally on port 4040. Its **Jobs**, **Stages**, and **Executors** tabs provide more detail.

## 2. Create an RDD

An RDD is a distributed collection. `parallelize` divides the input values into partitions. Here we explicitly request three partitions.

In [ ]:
numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
numbers_rdd = sc.parallelize(numbers, numSlices=3)

print("Number of partitions:", numbers_rdd.getNumPartitions())
print("Values by partition:", numbers_rdd.glom().collect())

`glom()` changes each partition into a Python list. `glom()` is a transformation. `collect()` is the action that starts the job and returns the partition lists to the driver.

The values may not be divided equally because ten values cannot be split evenly across three partitions.

## 3. Transformations are lazy

A transformation describes a new RDD. Spark does not process the data when the transformation is declared.

In [ ]:
even_rdd = numbers_rdd.filter(lambda number: number % 2 == 0)
squared_rdd = even_rdd.map(lambda number: number * number)

print("Transformations created. No action has run yet.")

The transformation chain is:

```text
numbers_rdd -> filter even numbers -> square each number
```

`filter` can remove records. `map` produces one output for every input it receives. Both are narrow transformations here, so a record can be processed without moving data between partitions.

## 4. Actions start jobs

Run each line separately and watch the **Jobs** tab in the Spark application UI. Each action normally starts a separate Spark job.

In [ ]:
sc.setJobGroup("collect-squares", "Collect squared even numbers")
print("collect:", squared_rdd.collect())

In [ ]:
sc.setJobGroup("count-squares", "Count squared even numbers")
print("count:", squared_rdd.count())

In [ ]:
sc.setJobGroup("take-squares", "Take the first three squared even numbers")
print("take:", squared_rdd.take(3))

In [ ]:
sc.setJobGroup("sum-squares", "Add the squared even numbers")
print("reduce:", squared_rdd.reduce(lambda left, right: left + right))

Common actions:

- `collect()` returns every result to the driver. Use it only for small results.
- `count()` returns the number of records.
- `take(n)` returns at most the first `n` records.
- `reduce(function)` combines the records into one result.

## 5. Partitions and tasks

A partition is a unit of distributed data. A task is a unit of work executed for one partition in one stage. Therefore, a stage that processes three partitions normally starts three tasks.

Use `mapPartitionsWithIndex` to see which values are processed in each partition.

In [ ]:
def describe_partition(partition_id, values):
    values = list(values)
    yield {
        "partition_id": partition_id,
        "values": values,
        "record_count": len(values),
    }

sc.setJobGroup("inspect-partitions", "Inspect records in each partition")
partition_details = numbers_rdd.mapPartitionsWithIndex(describe_partition).collect()

for detail in partition_details:
    print(detail)

Open the job in the Spark UI, then open its stage. Compare **Number of Tasks** with `numbers_rdd.getNumPartitions()`. For this simple narrow transformation, they should match.

A worker can run multiple tasks. A partition is data; a task is work performed on that partition.

## 6. Changing the number of partitions

`repartition` redistributes records and can increase or decrease the partition count. Redistribution is called a shuffle and introduces a stage boundary.

In [ ]:
repartitioned_rdd = squared_rdd.repartition(4)

print("Before repartition:", squared_rdd.getNumPartitions())
print("After repartition :", repartitioned_rdd.getNumPartitions())

sc.setJobGroup("repartition-demo", "Observe shuffle stages and tasks")
print("Values by partition:", repartitioned_rdd.glom().collect())

Open this job in the Spark UI. It should contain more than one stage because `repartition` requires a shuffle. Each stage has its own tasks and partition count. Empty partitions are possible with a very small dataset.

## 7. RDD lineage

Lineage is the sequence of transformations used to build an RDD. Spark keeps this information so it can plan work and recompute a lost partition.

In [ ]:
print(squared_rdd.toDebugString().decode("utf-8"))

In [ ]:
print(repartitioned_rdd.toDebugString().decode("utf-8"))

Look for `ShuffledRDD` in the second lineage. It marks the shuffle created by `repartition`. The indentation shows RDD dependencies, not completed execution. An action is still required to run the lineage.

## 8. Job, stage, task, and partition summary

- **Application:** this notebook's connection to Spark, represented by its `SparkContext`.
- **Job:** work started by an action such as `collect` or `count`.
- **Stage:** a group of tasks that can run without crossing a shuffle boundary.
- **Task:** work on one partition during one stage.
- **Partition:** one logical piece of an RDD.
- **Lineage:** the dependency plan created by transformations.

A useful mental model is: an action creates a job, a job contains stages, and a stage runs one task per partition.

## 9. Practice

Create an RDD containing the numbers 1 through 20 with four partitions. Then:

1. Keep numbers divisible by 3.
2. Multiply each remaining number by 10.
3. Inspect the partitions with `glom().collect()`.
4. Count the results.
5. Print the RDD lineage.
6. Identify the jobs and tasks in the Spark UI.

In [ ]:
# Write the practice solution here.


## 10. Stop Spark

Run this cell when the notebook is complete.

In [ ]:
spark.stop()
print("Spark session stopped.")